# CAISO OASIS Market Analysis

Interactive visualization and analysis of real CAISO market data.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.oasis import CAISOClient
from src.economics import analyze_caiso_lmp_data

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

## View Existing Visualizations

In [ ]:
plots_dir = Path('../data/plots')

print("LMP Components Breakdown:")
display(Image(filename=str(plots_dir / 'lmp_components.png')))

In [ ]:
print("Trading Hub Price Comparison:")
display(Image(filename=str(plots_dir / 'trading_hubs.png')))

In [ ]:
print("Fuel Mix Generation Stack:")
display(Image(filename=str(plots_dir / 'fuel_mix_stack.png')))

In [ ]:
print("Average Fuel Mix:")
display(Image(filename=str(plots_dir / 'fuel_mix_pie.png')))

In [ ]:
print("System Load Profile:")
display(Image(filename=str(plots_dir / 'load_profile.png')))

## Fetch Fresh Data and Analyze

In [ ]:
# Fetch latest data
end = datetime.now()
start = end - timedelta(hours=24)

with CAISOClient() as client:
    lmp_data = client.get_lmp(start, end, market="RTM")
    
print(f"Retrieved {len(lmp_data)} LMP records")
lmp_data.head()

In [ ]:
# Analyze LMP components
analysis = analyze_caiso_lmp_data(lmp_data)

print("\nLMP Distribution:")
for key, value in analysis['distribution'].items():
    print(f"  {key}: ${value:.2f}/MWh" if isinstance(value, float) else f"  {key}: {value}")

print("\nComponent Contributions:")
for component, stats in analysis['components'].items():
    print(f"  {component}: ${stats['mean']:.2f}/MWh ({stats['contribution_pct']:.1f}%)")

In [ ]:
# Plot average LMP by hour
lmp_data['hour'] = pd.to_datetime(lmp_data['timestamp']).dt.hour
hourly_avg = lmp_data.groupby('hour')['lmp_total'].mean()

plt.figure(figsize=(12, 5))
plt.bar(hourly_avg.index, hourly_avg.values, color='#3498db', alpha=0.7)
plt.xlabel('Hour of Day', fontsize=12, fontweight='bold')
plt.ylabel('Average LMP ($/MWh)', fontsize=12, fontweight='bold')
plt.title('Average LMP by Hour of Day', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Refresh All Visualizations

Run this cell to regenerate all plots with fresh data:

In [ ]:
# This will fetch new data and regenerate all plots
%run visualize_data.py